Construção da camada Bronze com PySpark

In [ ]:
import os
import platform
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, lit

In [ ]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('construcao_camada_bronze')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('Versão do Spark:', spark.version)

Localização das pastas

In [ ]:
pasta_atual = Path.cwd()

if pasta_atual.name == 'notebooks':
    raiz_projeto = pasta_atual.parent
else:
    raiz_projeto = pasta_atual

pasta_raw = raiz_projeto / 'data' / 'arquivos_raw'
pasta_bronze = raiz_projeto / 'data' / 'bronze'

print('Pasta raw:', pasta_raw)
print('Pasta Bronze:', pasta_bronze)

Arquivos de origem


In [ ]:
fontes = {
    'uf': 'br_inep_avaliacao_alfabetizacao_uf.csv.gz',
    'meta_alfabetizacao_brasil': 'br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv.gz',
    'meta_alfabetizacao_uf': 'br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv.gz',
    'meta_alfabetizacao_municipio': 'br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv.gz',
    'municipio': 'br_inep_avaliacao_alfabetizacao_municipio.csv.gz',
    'alunos': 'br_inep_avaliacao_alfabetizacao_aluno.csv.gz',
}

for nome_tabela, nome_arquivo in fontes.items():
    print(nome_tabela, '->', nome_arquivo)

Verificação dos arquivos

In [ ]:
for nome_arquivo in fontes.values():
    caminho = pasta_raw / nome_arquivo

    if not caminho.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho}')

print('Os seis arquivos foram encontrados.')

Identificação da ingestão

In [ ]:
id_ingestao = str(uuid4())
data_ingestao = datetime.now().isoformat(timespec='seconds')

print('ID da ingestão:', id_ingestao)
print('Data da ingestão:', data_ingestao)

Criação de um DataFrame Bronze

In [ ]:
def criar_dataframe_bronze(nome_tabela, nome_arquivo):
    caminho = pasta_raw / nome_arquivo

    dataframe_raw = (
        spark.read
        .option('header', True)
        .option('inferSchema', False)
        .option('encoding', 'UTF-8')
        .csv(caminho.as_posix())
    )

    dataframe_bronze = (
        dataframe_raw
        .withColumn('_id_ingestao', lit(id_ingestao))
        .withColumn('_data_ingestao', lit(data_ingestao))
        .withColumn('_arquivo_origem', input_file_name())
        .withColumn('_tabela_origem', lit(nome_tabela))
    )

    return dataframe_bronze

Criação dos seis DataFrames

In [ ]:
dados_bronze = {}

for nome_tabela, nome_arquivo in fontes.items():
    dados_bronze[nome_tabela] = criar_dataframe_bronze(nome_tabela, nome_arquivo)
    print('DataFrame criado:', nome_tabela)

Conferência antes da gravação

In [ ]:
for nome_tabela, dataframe in dados_bronze.items():
    print('Tabela Bronze:', nome_tabela)
    dataframe.show(2, truncate=False)
    dataframe.printSchema()

Gravação em Parquet

In [ ]:
hadoop_home = os.getenv('HADOOP_HOME')
winutils_configurado = False

if hadoop_home:
    winutils_configurado = (Path(hadoop_home) / 'bin' / 'winutils.exe').exists()

usar_gravacao_spark = platform.system() != 'Windows' or winutils_configurado

if usar_gravacao_spark:
    print('Modo de gravação: PySpark')
else:
    print('Modo de gravação: Pandas/PyArrow para compatibilidade com Windows')

In [ ]:
def gravar_parquet_no_windows(nome_tabela, nome_arquivo, caminho_saida):
    caminho_saida.mkdir(parents=True, exist_ok=True)
    caminho_parquet = caminho_saida / 'parte-00000.parquet'
    escritor = None

    try:
        partes_csv = pd.read_csv(
            pasta_raw / nome_arquivo,
            compression='gzip',
            dtype='string',
            chunksize=200000,
        )

        for parte in partes_csv:
            parte['_id_ingestao'] = id_ingestao
            parte['_data_ingestao'] = data_ingestao
            parte['_arquivo_origem'] = str(pasta_raw / nome_arquivo)
            parte['_tabela_origem'] = nome_tabela

            tabela_arrow = pa.Table.from_pandas(parte, preserve_index=False)

            if escritor is None:
                escritor = pq.ParquetWriter(caminho_parquet, tabela_arrow.schema)

            escritor.write_table(tabela_arrow)
    finally:
        if escritor is not None:
            escritor.close()

In [ ]:
caminhos_bronze = {}

for nome_tabela, dataframe in dados_bronze.items():
    caminho_saida = pasta_bronze / nome_tabela / f'id_ingestao={id_ingestao}'
    nome_arquivo = fontes[nome_tabela]

    if usar_gravacao_spark:
        (
            dataframe.write
            .mode('overwrite')
            .parquet(caminho_saida.as_posix())
        )
    else:
        gravar_parquet_no_windows(nome_tabela, nome_arquivo, caminho_saida)

    caminhos_bronze[nome_tabela] = caminho_saida
    print('Tabela gravada:', nome_tabela)

Validação da gravação

In [ ]:
resultado_validacao = []

for nome_tabela, dataframe in dados_bronze.items():
    quantidade_origem = dataframe.count()

    if usar_gravacao_spark:
        quantidade_bronze = spark.read.parquet(
            caminhos_bronze[nome_tabela].as_posix()
        ).count()
    else:
        arquivo_parquet = caminhos_bronze[nome_tabela] / 'parte-00000.parquet'
        quantidade_bronze = pq.ParquetFile(arquivo_parquet).metadata.num_rows

    quantidades_iguais = quantidade_origem == quantidade_bronze

    resultado_validacao.append((
        nome_tabela,
        quantidade_origem,
        quantidade_bronze,
        quantidades_iguais,
    ))

colunas_resultado = ['tabela', 'registros_origem', 'registros_bronze', 'quantidades_iguais']
df_validacao = spark.createDataFrame(resultado_validacao, colunas_resultado)
df_validacao.show(truncate=False)

leitura da Bronze

In [ ]:
if usar_gravacao_spark:
    df_alunos_bronze = spark.read.parquet(caminhos_bronze['alunos'].as_posix())
    df_alunos_bronze.show(5, truncate=False)
else:
    arquivo_alunos = caminhos_bronze['alunos'] / 'parte-00000.parquet'
    arquivo_alunos_parquet = pq.ParquetFile(arquivo_alunos)
    amostra_alunos = arquivo_alunos_parquet.read_row_group(0).slice(0, 5)
    display(amostra_alunos.to_pandas())

In [ ]:
spark.stop()